# YOLOv8m PPE 安全装备检测 — 本地训练 Notebook
---
**数据集**: archive (17,264 张, 6 类别)  
**类别**: boots / gloves / goggles / helmet / person / vest  
**GPU**: NVIDIA RTX 3050 (4GB)  
**模型**: YOLOv8m (26M 参数, COCO 预训练, HF 镜像下载)  
**框架**: Ultralytics YOLOv8

## 1. 环境验证

In [ ]:
import torch
import os, sys
from pathlib import Path

print("=" * 60)
print("1. 环境验证")
print("=" * 60)
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 版本:    {torch.version.cuda}")
print(f"CUDA 可用:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    vram_gb = gpu_props.total_memory / 1e9
    print(f"GPU 型号:     {gpu_name}")
    print(f"GPU 显存:     {vram_gb:.1f} GB")
    print(f"计算能力:     {gpu_props.major}.{gpu_props.minor}")
    
    # CUDA 兼容性测试
    try:
        test_tensor = torch.randn(1000, 1000).cuda()
        _ = torch.matmul(test_tensor, test_tensor)
        print("✅ CUDA 兼容性测试通过")
    except Exception as e:
        print(f"❌ CUDA 测试失败: {e}")
        os.environ['CUDA_VISIBLE_DEVICES'] = ''
else:
    print("⚠️ CUDA 不可用，将使用 CPU 训练（速度较慢）")

## 2. 导入依赖

In [ ]:
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Noto Sans SC']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 所有依赖导入成功")
print(f"Ultralytics 版本: {YOLO.__module__.split('.')[0]}")

## 3. 数据集配置

In [ ]:
print("=" * 60)
print("3. 数据集配置")
print("=" * 60)

# 使用项目本地的 archive 数据集
BASE_DIR = Path.cwd()  # 当前工作目录 = 项目根目录
ARCHIVE_DIR = BASE_DIR / 'archive'
OUTPUT_DIR = BASE_DIR / 'train_output'
OUTPUT_DIR.mkdir(exist_ok=True)

# 读取原始 data.yaml
original_yaml_path = ARCHIVE_DIR / 'data.yaml'
print(f"数据配置文件: {original_yaml_path}")

with open(original_yaml_path, 'r', encoding='utf-8') as f:
    original_config = yaml.safe_load(f)

CLASS_NAMES = original_config.get('names', 
    ['boots', 'gloves', 'goggles', 'helmet', 'person', 'vest'])
NUM_CLASSES = original_config.get('nc', 6)

print(f"类别数量: {NUM_CLASSES}")
print(f"类别名称: {CLASS_NAMES}")

# 验证目录结构
for split in ['train', 'valid', 'test']:
    img_dir = ARCHIVE_DIR / split / 'images'
    lbl_dir = ARCHIVE_DIR / split / 'labels'
    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
    print(f"  {split}: {len(imgs):,} 图片, {len(lbls):,} 标注")

total_imgs = sum(
    len(list((ARCHIVE_DIR / s / 'images').glob('*.jpg')))
    for s in ['train', 'valid', 'test']
)
print(f"\n数据集总计: {total_imgs:,} 张图片")

## 4. 创建训练用 YAML 配置

In [ ]:
print("=" * 60)
print("4. 创建训练 YAML")
print("=" * 60)

# 构建绝对路径的 YAML 配置（ultralytics 需要）
train_yaml_path = BASE_DIR / 'dataset_train.yaml'

yaml_config = {
    'path': str(ARCHIVE_DIR.absolute()),
    'train': str((ARCHIVE_DIR / 'train' / 'images').absolute()),
    'val':   str((ARCHIVE_DIR / 'valid' / 'images').absolute()),
    'test':  str((ARCHIVE_DIR / 'test'  / 'images').absolute()),
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES,
}

with open(train_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_config, f, default_flow_style=False, allow_unicode=True)

print("训练 YAML 配置:")
print("-" * 40)
with open(train_yaml_path, 'r', encoding='utf-8') as f:
    print(f.read())
print(f"\nYAML 已保存到: {train_yaml_path}")

## 5. 验证数据集完整性

In [ ]:
print("=" * 60)
print("5. 数据集完整性验证")
print("=" * 60)

all_ok = True

for split in ['train', 'valid', 'test']:
    img_dir = ARCHIVE_DIR / split / 'images'
    lbl_dir = ARCHIVE_DIR / split / 'labels'
    
    # 检查图片和标注一一对应
    imgs = set(f.stem for f in img_dir.glob('*.jpg'))
    lbls = set(f.stem for f in lbl_dir.glob('*.txt'))
    
    imgs_without_labels = imgs - lbls
    labels_without_imgs = lbls - imgs
    
    print(f"\n{split}:")
    print(f"  图片: {len(imgs)} | 标注: {len(lbls)}")
    
    if imgs_without_labels:
        print(f"  ⚠️ {len(imgs_without_labels)} 张图片缺少标注")
        all_ok = False
    if labels_without_imgs:
        print(f"  ⚠️ {len(labels_without_imgs)} 个标注缺少对应图片")
        all_ok = False
    if not imgs_without_labels and not labels_without_imgs:
        print(f"  ✅ 图片和标注一一对应")

# 检查标注格式
print(f"\n标注格式检查:")
sample_lbl = list((ARCHIVE_DIR / 'train' / 'labels').glob('*.txt'))[0]
with open(sample_lbl, 'r') as f:
    lines = f.readlines()
print(f"  样本文件: {sample_lbl.name}")
print(f"  标注行数: {len(lines)}")
print(f"  首行内容: {lines[0].strip()[:80] if lines else '空'}")

# 检查类别ID是否在有效范围内
max_class_id = 0
for lbl_file in list((ARCHIVE_DIR / 'train' / 'labels').glob('*.txt'))[:1000]:
    with open(lbl_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                cid = int(float(parts[0]))
                max_class_id = max(max_class_id, cid)
print(f"  最大类别ID: {max_class_id} (预期: {NUM_CLASSES - 1})")
if max_class_id >= NUM_CLASSES:
    print(f"  ❌ 标注中存在超出范围的类别ID!")
    all_ok = False

if all_ok:
    print("\n✅ 数据集验证全部通过")
else:
    print("\n⚠️ 数据集存在问题，请检查后再训练")

## 6. 训练配置

In [ ]:
print("=" * 60)
print("6. 训练配置")
print("=" * 60)

use_cuda = torch.cuda.is_available()
device = 0 if use_cuda else 'cpu'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if use_cuda else 0

# ── YOLOv8m (Medium, 26M 参数) — 4GB 显存最佳平衡 ──
model_name = "yolov8m.pt"

# 4GB 显存下的批次设置（v8m 比 v8l 省一半显存）
if vram_gb >= 12:
    batch_size = 24
elif vram_gb >= 8:
    batch_size = 16
elif vram_gb >= 4:
    batch_size = 8        # RTX 3050 4GB — v8m 可以跑到 batch=8
else:
    batch_size = 4

print(f"GPU 显存: {vram_gb:.1f} GB")
print(f"选择模型: {model_name} (26M 参数)")
print(f"批次大小: {batch_size}")
print(f"训练设备: {'GPU (CUDA)' if use_cuda else 'CPU'}")
print(f"输入尺寸: 640x640")
print(f"训练轮数: 100")
print(f"优化器:   AdamW")
print(f"学习率:   0.001 (余弦退火)")
print(f"\nYOLOv8m 精度比 v8s 高 5-8%, 比 v8l 省一半显存, 4GB 显卡最佳选择")

## 7. 开始训练

In [ ]:
print("=" * 60)
print("7. 开始训练 (YOLOv8m + 镜像加速)")
print("=" * 60)

# ── 镜像加速：设置 HuggingFace 镜像（ultralytics 模型下载走 HF）──
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# 备用：如果模型已存在本地则跳过下载
MODEL_PATH = Path.cwd() / model_name
if MODEL_PATH.exists():
    print(f"✅ 模型已存在于本地: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")

if use_cuda:
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU 显存可用: {free_gb:.1f} GB")

# 加载预训练模型
print(f"\n加载预训练模型: {model_name}...")
model = YOLO(model_name)

# 训练参数
train_args = {
    'data': str(train_yaml_path),
    'epochs': 100,
    'batch': batch_size,
    'imgsz': 640,
    'device': device,
    'workers': 2 if use_cuda else 0,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'cos_lr': True,
    'close_mosaic': 10,
    'mosaic': 1.0,
    'mixup': 0.1,
    'save': True,
    'save_period': 10,
    'val': True,
    'plots': True,
    'patience': 20,
    'seed': 42,
    'exist_ok': True,
    'pretrained': True,
    'amp': use_cuda,
    'project': str(OUTPUT_DIR),
    'name': 'ppe_detection',
}

training_successful = False

try:
    print("🚀 开始 YOLOv8m GPU 训练...")
    results = model.train(**train_args)
    training_successful = True
    print("✅ 训练完成！")
    
except RuntimeError as e:
    error_msg = str(e)
    print(f"❌ GPU 训练失败: {error_msg[:200]}")
    
    if 'out of memory' in error_msg.lower():
        print("\n💡 显存不足，尝试 batch=4...")
        train_args['batch'] = 4
        train_args['workers'] = 1
        
        torch.cuda.empty_cache()
        try:
            model = YOLO(model_name)
            results = model.train(**train_args)
            training_successful = True
            print("✅ 降低批次后训练成功！")
        except Exception as e2:
            print(f"❌ 仍然失败: {e2}")
    
    elif use_cuda:
        print("\n切换到 CPU 训练...")
        train_args['device'] = 'cpu'
        train_args['batch'] = 4
        train_args['amp'] = False
        train_args['workers'] = 0
        
        try:
            model = YOLO(model_name)
            results = model.train(**train_args)
            training_successful = True
            print("✅ CPU 训练完成！")
        except Exception as e2:
            print(f"❌ CPU 训练也失败: {e2}")
    
except Exception as e:
    print(f"❌ 训练异常: {e}")

## 8. 训练结果

In [ ]:
if training_successful:
    print("=" * 60)
    print("8. 训练结果")
    print("=" * 60)
    
    results_dir = OUTPUT_DIR / 'ppe_detection'
    
    # ── 读取训练指标 ──
    csv_file = results_dir / 'results.csv'
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip()
        
        # 查找 mAP 列
        map50_col = [c for c in df.columns if 'mAP50' in c and 'mAP50-95' not in c]
        map5095_col = [c for c in df.columns if 'mAP50-95' in c]
        
        print(f"\n训练指标 (共 {len(df)} epochs):")
        print("-" * 50)
        
        if map50_col:
            col = map50_col[0]
            best_val = df[col].max()
            best_epoch = df[col].idxmax() + 1
            print(f"最佳 mAP@0.5:       {best_val:.4f}  (Epoch {best_epoch})")
        
        if map5095_col:
            col = map5095_col[0]
            best_val = df[col].max()
            best_epoch = df[col].idxmax() + 1
            print(f"最佳 mAP@0.5:0.95:  {best_val:.4f}  (Epoch {best_epoch})")
        
        # 最后一轮的指标
        print(f"\n最终轮指标 (Epoch {len(df)}):")
        last_row = df.iloc[-1]
        for col in df.columns[:8]:
            print(f"  {col}: {last_row[col]:.4f}" if isinstance(last_row[col], float) else f"  {col}: {last_row[col]}")
    
    # ── 模型文件 ──
    weights_dir = results_dir / 'weights'
    if weights_dir.exists():
        print(f"\n模型文件:")
        for w in sorted(weights_dir.glob('*.pt')):
            size_mb = os.path.getsize(w) / 1e6
            print(f"  {w.name:30s} ({size_mb:.1f} MB)")
        
        # 复制最佳模型到项目根目录
        best_pt = weights_dir / 'best.pt'
        if best_pt.exists():
            import shutil
            dest = BASE_DIR / 'best_trained.pt'
            shutil.copy(best_pt, dest)
            print(f"\n✅ 最佳模型已复制到: {dest}")
        
        last_pt = weights_dir / 'last.pt'
        if last_pt.exists():
            import shutil
            dest = BASE_DIR / 'last_trained.pt'
            shutil.copy(last_pt, dest)
            print(f"✅ 最终模型已复制到: {dest}")
    
    print(f"\n📁 完整训练输出: {results_dir}")

else:
    print("⚠️ 训练未成功完成")

## 9. 训练曲线可视化

In [ ]:
if training_successful:
    results_dir = OUTPUT_DIR / 'ppe_detection'
    csv_file = results_dir / 'results.csv'
    
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip()
        
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        
        # Box loss
        box_cols = [c for c in df.columns if 'box_loss' in c.lower()]
        for col in box_cols:
            label = 'train' if 'train' in col.lower() else 'val'
            axes[0, 0].plot(df.index + 1, df[col], label=label, linewidth=1.5)
        axes[0, 0].set_title('Box Loss', fontsize=13)
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Class loss
        cls_cols = [c for c in df.columns if 'cls_loss' in c.lower()]
        for col in cls_cols:
            label = 'train' if 'train' in col.lower() else 'val'
            axes[0, 1].plot(df.index + 1, df[col], label=label, linewidth=1.5)
        axes[0, 1].set_title('Class Loss', fontsize=13)
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # DFL loss
        dfl_cols = [c for c in df.columns if 'dfl_loss' in c.lower()]
        for col in dfl_cols:
            label = 'train' if 'train' in col.lower() else 'val'
            axes[0, 2].plot(df.index + 1, df[col], label=label, linewidth=1.5)
        axes[0, 2].set_title('DFL Loss', fontsize=13)
        axes[0, 2].legend()
        axes[0, 2].grid(True, alpha=0.3)
        
        # mAP@0.5
        map50_cols = [c for c in df.columns if 'mAP50' in c and 'mAP50-95' not in c]
        for col in map50_cols:
            axes[1, 0].plot(df.index + 1, df[col], linewidth=2, color='#2196F3')
        axes[1, 0].set_title('mAP@0.5', fontsize=13)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].grid(True, alpha=0.3)
        
        # mAP@0.5:0.95
        map5095_cols = [c for c in df.columns if 'mAP50-95' in c]
        for col in map5095_cols:
            axes[1, 1].plot(df.index + 1, df[col], linewidth=2, color='#4CAF50')
        axes[1, 1].set_title('mAP@0.5:0.95', fontsize=13)
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].grid(True, alpha=0.3)
        
        # Precision & Recall
        prec_cols = [c for c in df.columns if 'precision' in c.lower()]
        rec_cols = [c for c in df.columns if 'recall' in c.lower()]
        for col in prec_cols:
            axes[1, 2].plot(df.index + 1, df[col], label='Precision', linewidth=1.5, color='#FF9800')
        for col in rec_cols:
            axes[1, 2].plot(df.index + 1, df[col], label='Recall', linewidth=1.5, color='#9C27B0')
        axes[1, 2].set_title('Precision & Recall', fontsize=13)
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3)
        
        plt.suptitle('YOLOv8 PPE 训练曲线', fontsize=16, fontweight='bold', y=1.01)
        plt.tight_layout()
        
        # 保存
        save_path = results_dir / 'training_curves.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"📈 训练曲线已保存: {save_path}")
        plt.show()
    else:
        print("⚠️ 未找到 results.csv")

## 10. 模型验证
在测试集上评估最佳模型

In [ ]:
if training_successful:
    print("=" * 60)
    print("10. 测试集验证")
    print("=" * 60)
    
    weights_dir = OUTPUT_DIR / 'ppe_detection' / 'weights'
    best_pt = weights_dir / 'best.pt'
    
    if best_pt.exists():
        print(f"加载最佳模型: {best_pt}")
        model_val = YOLO(str(best_pt))
        
        # 在测试集上验证
        test_path = str((ARCHIVE_DIR / 'test' / 'images').absolute())
        print(f"测试集路径: {test_path}")
        print("\n运行验证...")
        
        metrics = model_val.val(
            data=str(train_yaml_path),
            split='test',
            device=device,
            verbose=True,
        )
        
        print(f"\n测试集结果:")
        print(f"  mAP@0.5:       {metrics.box.map50:.4f}")
        print(f"  mAP@0.5:0.95:  {metrics.box.map:.4f}")
        print(f"  Precision:     {metrics.box.mp:.4f}")
        print(f"  Recall:        {metrics.box.mr:.4f}")
    else:
        print(f"❌ 未找到最佳模型: {best_pt}")

## 11. 推理测试
用训练好的模型在测试图片上可视化检测结果

In [ ]:
if training_successful:
    print("=" * 60)
    print("11. 推理测试")
    print("=" * 60)
    
    weights_dir = OUTPUT_DIR / 'ppe_detection' / 'weights'
    best_pt = weights_dir / 'best.pt'
    
    if best_pt.exists():
        model_infer = YOLO(str(best_pt))
        
        # 随机选几张测试图
        test_imgs = list((ARCHIVE_DIR / 'test' / 'images').glob('*.jpg'))[:6]
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, img_path in enumerate(test_imgs):
            results = model_infer(str(img_path), conf=0.3, device=device, verbose=False)
            annotated = results[0].plot()
            
            axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            axes[idx].set_title(img_path.name[:40], fontsize=9)
            axes[idx].axis('off')
        
        plt.suptitle('训练模型推理结果', fontsize=15, fontweight='bold')
        plt.tight_layout()
        
        save_path = OUTPUT_DIR / 'ppe_detection' / 'inference_samples.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f"📸 推理结果已保存: {save_path}")
        plt.show()
        
        import cv2
    else:
        print(f"❌ 未找到模型: {best_pt}")

print("\n" + "=" * 60)
print("✅ Notebook 执行完毕")
print("=" * 60)